# 💼 Salary Predictor — Machine Learning Project
**Company Project Submission | Built with Python + scikit-learn**

This notebook is a direct translation of the IBM Bob project files:
- `generate_data.py` → **Cell 2** (Dataset Generation)
- `train_model.py` → **Cells 3–8** (Cleaning → EDA → Training → Evaluation → Saving)
- Prediction test → **Cell 9**
- Download models → **Cell 10**

---
**Run all cells in order: Runtime → Run all  (or Shift+Enter cell by cell)**

## Cell 1 — Install & Import Libraries
Google Colab already has numpy, pandas, matplotlib, seaborn, scikit-learn, and joblib.
This cell confirms versions and imports everything needed.

In [ ]:
# ── Cell 1: Install & Import Libraries ────────────────────────────────────────
# All libraries below are pre-installed in Google Colab.
# No pip install required.

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

# Create output directories (mirrors project structure)
os.makedirs('data',   exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('charts', exist_ok=True)

# Version check
import sklearn
print('numpy   :', np.__version__)
print('pandas  :', pd.__version__)
print('sklearn :', sklearn.__version__)
print('joblib  :', joblib.__version__)
print()
print('✅ All libraries loaded successfully.')

## Cell 2 — Dataset Generation
Translated directly from `generate_data.py`. Generates 2,000 synthetic salary records
with education, job title, location, experience, age, and annual salary.
Intentionally includes ~2% missing values and ~1% duplicates for realistic cleaning.

In [ ]:
# ── Cell 2: Dataset Generation (from generate_data.py) ────────────────────────
np.random.seed(42)
N = 2000

education_levels = ['High School', 'Associate', 'Bachelor', 'Master', 'PhD']
job_titles = [
    'Software Engineer', 'Data Analyst', 'Data Scientist', 'Product Manager',
    'Marketing Specialist', 'Financial Analyst', 'HR Manager', 'Sales Representative',
    'UX Designer', 'DevOps Engineer', 'Business Analyst', 'Project Manager',
]
locations = [
    'New York', 'San Francisco', 'Austin', 'Chicago', 'Seattle',
    'Boston', 'Denver', 'Los Angeles', 'Miami', 'Dallas'
]

# Base salary per education level (USD)
edu_base = {
    'High School': 32000,
    'Associate':   40000,
    'Bachelor':    58000,
    'Master':      75000,
    'PhD':         92000,
}

# Job title salary multiplier
job_mult = {
    'Software Engineer':    1.35,
    'Data Analyst':         1.15,
    'Data Scientist':       1.45,
    'Product Manager':      1.40,
    'Marketing Specialist': 0.95,
    'Financial Analyst':    1.20,
    'HR Manager':           1.05,
    'Sales Representative': 0.90,
    'UX Designer':          1.10,
    'DevOps Engineer':      1.30,
    'Business Analyst':     1.12,
    'Project Manager':      1.25,
}

# Location cost-of-living multiplier
loc_mult = {
    'New York':      1.30,
    'San Francisco': 1.45,
    'Austin':        1.05,
    'Chicago':       1.15,
    'Seattle':       1.25,
    'Boston':        1.20,
    'Denver':        1.10,
    'Los Angeles':   1.25,
    'Miami':         1.05,
    'Dallas':        1.00,
}

records = []
for _ in range(N):
    edu = np.random.choice(education_levels, p=[0.12, 0.13, 0.40, 0.25, 0.10])
    job = np.random.choice(job_titles)
    loc = np.random.choice(locations)
    exp = float(np.random.randint(0, 31))                          # 0–30 years
    age = int(max(22, min(65, exp + np.random.randint(20, 28))))   # derived from exp

    base_salary = (
        edu_base[edu]
        * job_mult[job]
        * loc_mult[loc]
        + exp * 2200                          # +$2,200 per year of experience
        + np.random.normal(0, 6000)           # random noise
    )
    base_salary = round(max(25000, base_salary), 2)

    records.append({
        'age':        age,
        'education':  edu,
        'job_title':  job,
        'location':   loc,
        'experience': exp,
        'salary':     base_salary,
    })

df_raw = pd.DataFrame(records)

# Inject ~2% missing values in numeric columns
for col in ['experience', 'age']:
    mask = np.random.choice([True, False], size=N, p=[0.02, 0.98])
    df_raw.loc[mask, col] = np.nan

# Inject ~1% duplicate rows
dup_idx = np.random.choice(df_raw.index, size=int(N * 0.01), replace=False)
df_raw = pd.concat([df_raw, df_raw.loc[dup_idx]], ignore_index=True)

# Save to CSV (mirrors data/salary_data.csv from the project)
df_raw.to_csv('data/salary_data.csv', index=False)

print(f'Dataset created : {len(df_raw):,} rows × {df_raw.shape[1]} columns')
print(f'Columns         : {list(df_raw.columns)}')
print(f'Missing values  :\n{df_raw.isnull().sum().to_string()}')
print()
df_raw.head()

## Cell 3 — Data Loading & Cleaning
Translated from the first section of `train_model.py`.
Drops duplicates, fills missing numeric values with the median, and enforces correct data types.

In [ ]:
# ── Cell 3: Data Loading & Cleaning (train_model.py §1) ───────────────────────
df = pd.read_csv('data/salary_data.csv')
print(f'Raw shape : {df.shape}')

# ── Drop duplicates ───────────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed : {before - len(df)}')

# ── Fill missing numeric values with column median ────────────────────────────
for col in ['experience', 'age']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    df[col] = df[col].astype(float)

# ── Enforce correct types ─────────────────────────────────────────────────────
df['salary']     = pd.to_numeric(df['salary'],     errors='coerce')
df['experience'] = pd.to_numeric(df['experience'], errors='coerce')
df['age']        = pd.to_numeric(df['age'],        errors='coerce')
df.dropna(subset=['salary'], inplace=True)

print(f'Clean shape : {df.shape}')
print(f'Remaining nulls:\n{df.isnull().sum().to_string()}')
print()
print('Salary statistics:')
display(df['salary'].describe().to_frame().T.style.format('${:,.2f}'))

## Cell 4 — Exploratory Data Analysis (EDA)
Translated from `train_model.py §2`. Charts render inline in Colab instead of being saved to disk
(they are also saved to `charts/` so you can download them).

In [ ]:
# ── Cell 4: Exploratory Data Analysis (train_model.py §2) ─────────────────────

# ── 4a. Salary Distribution ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['salary'], bins=40, kde=True, color='#3b82d4', ax=ax)
ax.set_title('Annual Salary Distribution', fontsize=14)
ax.set_xlabel('Annual Salary (USD)')
ax.set_ylabel('Count')
plt.tight_layout()
fig.savefig('charts/salary_distribution.png', dpi=120)
plt.show()

# ── 4b. Salary by Education Level ─────────────────────────────────────────────
edu_order = ['High School', 'Associate', 'Bachelor', 'Master', 'PhD']
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x='education', y='salary', order=edu_order,
            palette='Blues', ax=ax)
ax.set_title('Salary by Education Level', fontsize=14)
ax.set_xlabel('Education Level')
ax.set_ylabel('Annual Salary (USD)')
plt.tight_layout()
fig.savefig('charts/salary_by_education.png', dpi=120)
plt.show()

# ── 4c. Salary vs Years of Experience ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
sns.scatterplot(data=df, x='experience', y='salary', alpha=0.4,
                color='#7c5cd8', ax=ax)
sns.regplot(data=df, x='experience', y='salary',
            scatter=False, color='red', ax=ax)
ax.set_title('Salary vs Years of Experience', fontsize=14)
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Annual Salary (USD)')
plt.tight_layout()
fig.savefig('charts/salary_vs_experience.png', dpi=120)
plt.show()

# ── 4d. Median Salary by Job Title ────────────────────────────────────────────
med_job = df.groupby('job_title')['salary'].median().sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
med_job.plot(kind='barh', color='#3b82d4', ax=ax)
ax.set_title('Median Salary by Job Title', fontsize=14)
ax.set_xlabel('Median Annual Salary (USD)')
plt.tight_layout()
fig.savefig('charts/salary_by_job.png', dpi=120)
plt.show()

# ── 4e. Median Salary by Location ─────────────────────────────────────────────
med_loc = df.groupby('location')['salary'].median().sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
med_loc.plot(kind='barh', color='#7c5cd8', ax=ax)
ax.set_title('Median Salary by Location', fontsize=14)
ax.set_xlabel('Median Annual Salary (USD)')
plt.tight_layout()
fig.savefig('charts/salary_by_location.png', dpi=120)
plt.show()

# ── 4f. Correlation heatmap (numeric features) ────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(df[['age', 'experience', 'salary']].corr(),
            annot=True, fmt='.2f', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
fig.savefig('charts/correlation_heatmap.png', dpi=120)
plt.show()

# ── Key observations ──────────────────────────────────────────────────────────
corr_exp = df['experience'].corr(df['salary'])
print('─' * 50)
print('Key EDA Observations')
print('─' * 50)
print(f'  Median salary          : ${df["salary"].median():>10,.0f}')
print(f'  Mean salary            : ${df["salary"].mean():>10,.0f}')
print(f'  Highest median job     : {med_job.idxmax()}')
print(f'  Lowest  median job     : {med_job.idxmin()}')
print(f'  Highest median location: {med_loc.idxmax()}')
print(f'  Lowest  median location: {med_loc.idxmin()}')
print(f'  Experience ↔ Salary r  : {corr_exp:.3f}')

## Cell 5 — Feature Engineering & Preprocessing Pipeline
Translated from `train_model.py §3`.
Numerical features (age, experience) → `StandardScaler`.
Categorical features (education, job_title, location) → `OrdinalEncoder`.
Both are combined in a `ColumnTransformer` inside a scikit-learn `Pipeline`.

In [ ]:
# ── Cell 5: Feature Engineering & Preprocessing (train_model.py §3) ───────────
FEATURES = ['age', 'education', 'job_title', 'location', 'experience']
TARGET   = 'salary'

X = df[FEATURES]
y = df[TARGET]

CAT_COLS = ['education', 'job_title', 'location']
NUM_COLS = ['age', 'experience']

# Fixed category lists — must match app.py exactly
edu_categories = ['High School', 'Associate', 'Bachelor', 'Master', 'PhD']
job_categories = sorted(df['job_title'].unique().tolist())
loc_categories = sorted(df['location'].unique().tolist())

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OrdinalEncoder(
        categories=[edu_categories, job_categories, loc_categories],
        handle_unknown='use_encoded_value',
        unknown_value=-1,
    ), CAT_COLS),
], remainder='drop')

# 80/20 train-test split — same random state as train_model.py
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Features      : {FEATURES}')
print(f'Target        : {TARGET}')
print(f'Train samples : {len(X_train)}')
print(f'Test  samples : {len(X_test)}')
print(f'Education categories : {edu_categories}')
print(f'Job categories       : {job_categories}')
print(f'Location categories  : {loc_categories}')

## Cell 6 — Model Training
Translated from `train_model.py §4`.
Trains **Linear Regression** and **Random Forest Regressor** (200 trees) inside the same pipeline.
Both models share the identical preprocessing steps.

In [ ]:
# ── Cell 6: Model Training (train_model.py §4) ────────────────────────────────
models_to_train = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

results   = {}
pipelines = {}

for name, estimator in models_to_train.items():
    print(f'Training {name} …', end=' ', flush=True)
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model',        estimator),
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results[name]   = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    pipelines[name] = pipe
    print('done.')

print('\n✅ Training complete.')

## Cell 7 — Model Evaluation
Translated from `train_model.py §4–5`.
Prints MAE, RMSE, R² for both models and displays residual / actual-vs-predicted plots for the best model.

In [ ]:
# ── Cell 7: Model Evaluation (train_model.py §4–5) ────────────────────────────

# ── Metrics table ─────────────────────────────────────────────────────────────
print('=' * 52)
print(f'{"Model":<22} {"MAE":>9}  {"RMSE":>9}  {"R²":>7}')
print('=' * 52)
for name, res in results.items():
    print(f'{name:<22} ${res["MAE"]:>8,.2f}  ${res["RMSE"]:>8,.2f}  {res["R2"]:>7.4f}')
print('=' * 52)

# ── Select best model (highest R²) ────────────────────────────────────────────
best_name = max(results, key=lambda k: results[k]['R2'])
best_pipe = pipelines[best_name]
print(f'\n🏆 Best model: {best_name}  (R² = {results[best_name]["R2"]:.4f})')

# ── Residual & Actual vs Predicted plots ──────────────────────────────────────
y_pred_best = best_pipe.predict(X_test)
residuals   = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred_best, residuals, alpha=0.35, color='#3b82d4')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title(f'Residuals — {best_name}', fontsize=13)
axes[0].set_xlabel('Predicted Salary (USD)')
axes[0].set_ylabel('Residual (Actual − Predicted)')

axes[1].scatter(y_test, y_pred_best, alpha=0.35, color='#7c5cd8')
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_title('Actual vs Predicted Salary', fontsize=13)
axes[1].set_xlabel('Actual Salary (USD)')
axes[1].set_ylabel('Predicted Salary (USD)')
axes[1].legend()

plt.suptitle(f'Model Diagnostics — {best_name}', fontsize=15, y=1.02)
plt.tight_layout()
fig.savefig(f'charts/residuals_{best_name.replace(" ", "_")}.png', dpi=120)
plt.show()

# ── MAE / RMSE bar chart ──────────────────────────────────────────────────────
model_names = list(results.keys())
mae_vals    = [results[m]['MAE']  for m in model_names]
rmse_vals   = [results[m]['RMSE'] for m in model_names]
r2_vals     = [results[m]['R2']   for m in model_names]

fig, axes2 = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(len(model_names))
w = 0.35
axes2[0].bar(x - w/2, mae_vals,  w, label='MAE',  color='#3b82d4')
axes2[0].bar(x + w/2, rmse_vals, w, label='RMSE', color='#7c5cd8')
axes2[0].set_xticks(x)
axes2[0].set_xticklabels(model_names)
axes2[0].set_ylabel('Error (USD)')
axes2[0].set_title('MAE & RMSE Comparison')
axes2[0].legend()

colors = ['#22863a' if m == best_name else '#3b82d4' for m in model_names]
axes2[1].bar(model_names, r2_vals, color=colors)
axes2[1].set_ylabel('R² Score')
axes2[1].set_ylim(0, 1.05)
axes2[1].set_title('R² Score Comparison')
for i, v in enumerate(r2_vals):
    axes2[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=11)

plt.tight_layout()
fig.savefig('charts/model_comparison.png', dpi=120)
plt.show()

## Cell 8 — Save Models with Joblib
Translated from `train_model.py §6`. Saves both pipelines and the metadata dict.
These are the exact same files that `app.py` loads at startup.

In [ ]:
# ── Cell 8: Save Models (train_model.py §6) ────────────────────────────────────

# Save each pipeline
for name, pipe in pipelines.items():
    fname = name.replace(' ', '_').lower() + '_pipeline.joblib'
    path  = os.path.join('models', fname)
    joblib.dump(pipe, path)
    print(f'Saved: {path}')

# Save the best model under a fixed name (what app.py looks for)
joblib.dump(best_pipe, 'models/best_model.joblib')
print(f'Saved: models/best_model.joblib  (best = {best_name})')

# Save metadata (category lists + results) for the Streamlit app
metadata = {
    'best_model':     best_name,
    'edu_categories': edu_categories,
    'job_categories': job_categories,
    'loc_categories': loc_categories,
    'results':        results,
}
joblib.dump(metadata, 'models/metadata.joblib')
print('Saved: models/metadata.joblib')

# List saved files
print('\nFiles in models/:')
for f in sorted(os.listdir('models')):
    size_kb = os.path.getsize(f'models/{f}') / 1024
    print(f'  {f:45s}  {size_kb:7.1f} KB')

## Cell 9 — Salary Prediction Tests
Directly tests the saved pipeline with multiple real-world-style inputs.
This is the same prediction logic that `app.py` uses when the user clicks **Predict Salary**.

In [ ]:
# ── Cell 9: Salary Prediction Tests ───────────────────────────────────────────

# Reload from disk to confirm joblib round-trip works
loaded_model    = joblib.load('models/best_model.joblib')
loaded_metadata = joblib.load('models/metadata.joblib')
print(f'Loaded model    : {loaded_metadata["best_model"]}')
print(f'Education opts  : {loaded_metadata["edu_categories"]}')
print()

# Define test cases (mirrors the sanity check in train_model.py)
test_cases = [
    {
        'label':      '🎓 Junior Data Scientist, San Francisco',
        'age':        30,
        'education':  'Bachelor',
        'job_title':  'Data Scientist',
        'location':   'San Francisco',
        'experience': 5.0,
    },
    {
        'label':      '🏢 Senior Software Engineer, New York',
        'age':        40,
        'education':  'Master',
        'job_title':  'Software Engineer',
        'location':   'New York',
        'experience': 15.0,
    },
    {
        'label':      '📊 Entry HR Manager, Dallas',
        'age':        25,
        'education':  'Bachelor',
        'job_title':  'HR Manager',
        'location':   'Dallas',
        'experience': 2.0,
    },
    {
        'label':      '🔬 PhD Product Manager, Seattle',
        'age':        45,
        'education':  'PhD',
        'job_title':  'Product Manager',
        'location':   'Seattle',
        'experience': 20.0,
    },
    {
        'label':      '💼 Mid-level Financial Analyst, Chicago',
        'age':        33,
        'education':  'Master',
        'job_title':  'Financial Analyst',
        'location':   'Chicago',
        'experience': 8.0,
    },
]

print('─' * 70)
print(f'{"Profile":<45} {"Predicted Salary":>18}  {"Range (±10%)"}' )
print('─' * 70)

prediction_rows = []
for tc in test_cases:
    label = tc.pop('label')
    input_df  = pd.DataFrame([tc])
    predicted = loaded_model.predict(input_df)[0]
    low, high = predicted * 0.90, predicted * 1.10
    print(f'{label:<45} ${predicted:>14,.0f}  ${low:,.0f} – ${high:,.0f}')
    prediction_rows.append({
        'Profile':   label,
        'Education': tc['education'],
        'Job Title': tc['job_title'],
        'Location':  tc['location'],
        'Exp (yrs)': tc['experience'],
        'Predicted': f'${predicted:,.0f}',
    })

print('─' * 70)
print()
print('⚠️  Disclaimer: These are statistical estimates from a model trained on')
print('   synthetic data. They do not constitute salary offers or HR advice.')
print()
display(pd.DataFrame(prediction_rows).set_index('Profile'))

## Cell 10 — Download Model Files
Downloads the three `.joblib` files from Colab to your local machine.
Copy these into your `salary-predictor/models/` folder so `app.py` can load them directly.

In [ ]:
# ── Cell 10: Download MISSING Model Files ─────────────────────────────────────
# linear_regression_pipeline.joblib and salary_data.csv already downloaded OK.
# This cell downloads only the 3 files that are still missing.
from google.colab import files

missing_files = [
    'models/best_model.joblib',
    'models/random_forest_pipeline.joblib',
    'models/metadata.joblib',
]

for path in missing_files:
    if os.path.exists(path):
        files.download(path)
        print(f'⬇️  Downloading: {path}')
    else:
        print(f'⚠️  Not found — run Cells 1-8 first, then re-run this cell: {path}')

## Cell 11 — (Optional) Run Streamlit App Inside Colab
This gives you a public preview URL to screenshot for your Google Form.
It installs `streamlit` and `pyngrok`, uploads `app.py`, and tunnels the app.

**Note:** You need a free ngrok account token from [dashboard.ngrok.com](https://dashboard.ngrok.com).
Replace `YOUR_NGROK_TOKEN` below with your token.

In [ ]:
# ── Cell 11 (OPTIONAL): Run Streamlit app with a public ngrok URL ──────────────
# Skip this cell if you plan to deploy on Streamlit Community Cloud instead.

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'   # ← paste your free token here

# Install streamlit and ngrok
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'streamlit', 'pyngrok', '-q'], check=True)

# Upload app.py from your local machine
from google.colab import files
print('Please upload app.py from your salary-predictor/ folder:')
uploaded = files.upload()   # select app.py when the dialog opens

# Authenticate ngrok
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in the background
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

import time; time.sleep(4)   # wait for Streamlit to start

# Open the tunnel
public_url = ngrok.connect(8501)
print()
print('=' * 60)
print(f'🌐 Streamlit app is live at: {public_url}')
print('   Open this URL in your browser, take a screenshot,')
print('   and attach it to your Google Form submission.')
print('=' * 60)